#### Download Nvidia Annual Report

In [1]:
import os
import requests
from pathlib import Path


# Create data directory if it doesn't exist
data_folder = Path('..') / 'data'
os.makedirs(data_folder, exist_ok=True)

# Download the PDF file
url = 'https://s201.q4cdn.com/141608511/files/doc_financials/2025/annual/NVIDIA-2025-Annual-Report.pdf'
response = requests.get(url)

# Save the file to data directory
file_path = data_folder / 'NVIDIA-2025-Annual-Report.pdf'
with open(file_path, 'wb') as f:
    f.write(response.content)

print(f"File downloaded and saved to {file_path}")

File downloaded and saved to ..\data\NVIDIA-2025-Annual-Report.pdf


#### Convert to markdown for chunking

In [2]:
from markitdown import MarkItDown

# Initialize MarkItDown
converter = MarkItDown()

# Convert PDF to markdown
markdown_content = converter.convert(file_path)

#### Use simple langchain chunking

In [3]:
markdown_content = markdown_content.text_content

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Create a RecursiveCharacterTextSplitter with 384 max characters per chunk
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
)

chunks = text_splitter.create_documents([markdown_content])

c:\Users\neutr\.conda\envs\graph_rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Call the graphrag

In [5]:
import sys

# Add parent directory to path
parent_dir = os.path.dirname(os.getcwd())
sys.path.append(parent_dir)

In [6]:
from src.graph_rag import GLiNERGraphStore
from src.graph_rag import GLiNERGraphRetriever

c:\Users\neutr\.conda\envs\graph_rag\Lib\site-packages\pydantic\_internal\_config.py:386: UserWarning: Valid config keys have changed in V2:
* 'underscore_attrs_are_private' has been removed
  warnings.warn(message, UserWarning)


In [7]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI

In [8]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-bge-small-en-v1.5",  # or whatever LM Studio exposes
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",  # required field, but ignored by LM Studio,
    check_embedding_ctx_length=False
)

In [9]:
llm = ChatOpenAI(
    model="gemma-4-e4b-it",  # must match LM Studio model name
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    temperature=0.1,
)

In [10]:
vectorstore = InMemoryVectorStore(embedding=embeddings)

#### Auto retrieval

In [11]:
graph_store = GLiNERGraphRetriever(
    vectorstore=vectorstore,
    model_path = r"D:\Projects\graph_rag\models\knowledgator--gliner-relex-large-v1.0",
    collection_name="test_collection",
    labels=['organization'],
    relations=['supplies to', 'competitor of', 'supplied by']
)


In [12]:
_ = graph_store.add_documents(chunks[: 10])

Extracting graph edges: 100%|██████████| 10/10 [00:05<00:00,  1.88it/s]


In [13]:
graph_store.invoke('what does nvidia produce for gaming industry?')

[Document(id='741a820f-038c-442f-8941-247464f9030b', metadata={}, page_content='industries. Intelligence is no longer stored and retrieved. It’s continuously generated by\n\nAI factories. NVIDIA is building the global supply chain for this new essential resource.\n\n\x0cCUDA-X: Turning Acceleration\nInto Impact\n\nNVIDIA CUDA-X is the language of accelerated computing. It’s a layered software\n\nstack of domain-specific libraries, frameworks, and tools built on NVIDIA CUDA. It\n\nabstracts complexity and delivers performance across industries for applications\n\nspanning materials science, weather modeling, and industrial design. CUDA-X is how\n\naccelerated computing becomes accessible, powerful, and practical—transforming\n\nraw GPU capability into real-world breakthroughs.\n\nWarp\n\nPhysics\n\ncuDF\ncuML\n\nData Science\n\nand Processing\n\ncuEquivariance\ncuTENSOR\n\ncuQuantum\nCUDA-Q\n\nQuantum\nChemistry\n\nQuantum\nComputing\n\ncuDSS\ncuSPARSE\ncuFFT\nAmgX\n\nComputer-Aided\nEn

#### LLM Assisted

In [42]:
query = 'what does nvidia produce for gaming industry?'

In [43]:
seed_docs    = graph_store.vectorstore.similarity_search(query, k=graph_store.k)
seed_ids     = [doc.id for doc in seed_docs]

In [44]:
# Step 1 — collect candidate entities from seed documents
entities     = graph_store.get_entry_entities(seed_ids)

In [45]:
# Step 2 — LLM selects relevant entities
schema       = graph_store.build_entity_filter_schema(entities)
decision     = llm.with_structured_output(schema).invoke(query)
selected_ents = {e: t for e, t in entities.items()
                if getattr(decision, e.replace(' ', '_'), False)}

In [53]:
selected_ents

{'rtx ai pcs': 'organization',
 'enterprise': 'organization',
 'industry': 'organization',
 'nvidia': 'organization',
 'nvidia corporation': 'organization'}

In [46]:
# Step 3 — collect triples reachable from selected entities
triples      = graph_store.get_reachable_triples(selected_ents)

In [47]:
# Step 4 — LLM selects relevant triples
schema       = graph_store.build_triple_filter_schema(triples)
decision     = llm.with_structured_output(schema).invoke(query)
selected_triples = [t for t in triples
                    if getattr(decision, t.id.replace('-','_'), False)]

In [48]:
docs         = graph_store.resolve_docs_from_triples(selected_triples)

In [50]:
selected_triples

[TripleRecord(id='e64f3e67-08ef-4b36-bf02-69eb2b471b6a', text='nvidia (organization) → competitor of → nvidia corporation (organization)', head='nvidia', head_type='organization', relation='competitor of', tail='nvidia corporation', tail_type='organization', edge_source='0a6ec06b-be2f-4287-8e62-11e24d270a62'),
 TripleRecord(id='0b7eb4da-3f91-4b7f-bab6-b6815815691a', text='nvidia (organization) → supplies to → materials science (organization)', head='nvidia', head_type='organization', relation='supplies to', tail='materials science', tail_type='organization', edge_source='741a820f-038c-442f-8941-247464f9030b'),
 TripleRecord(id='3fbb1fab-309b-4f17-b260-4088039b82ad', text='nvidia (organization) → supplies to → industrial design (organization)', head='nvidia', head_type='organization', relation='supplies to', tail='industrial design', tail_type='organization', edge_source='741a820f-038c-442f-8941-247464f9030b'),
 TripleRecord(id='b7c08888-dc6c-48cd-8afb-87d7c0af799c', text='nvidia (organ

In [49]:
docs

[Document(id='741a820f-038c-442f-8941-247464f9030b', metadata={}, page_content='industries. Intelligence is no longer stored and retrieved. It’s continuously generated by\n\nAI factories. NVIDIA is building the global supply chain for this new essential resource.\n\n\x0cCUDA-X: Turning Acceleration\nInto Impact\n\nNVIDIA CUDA-X is the language of accelerated computing. It’s a layered software\n\nstack of domain-specific libraries, frameworks, and tools built on NVIDIA CUDA. It\n\nabstracts complexity and delivers performance across industries for applications\n\nspanning materials science, weather modeling, and industrial design. CUDA-X is how\n\naccelerated computing becomes accessible, powerful, and practical—transforming\n\nraw GPU capability into real-world breakthroughs.\n\nWarp\n\nPhysics\n\ncuDF\ncuML\n\nData Science\n\nand Processing\n\ncuEquivariance\ncuTENSOR\n\ncuQuantum\nCUDA-Q\n\nQuantum\nChemistry\n\nQuantum\nComputing\n\ncuDSS\ncuSPARSE\ncuFFT\nAmgX\n\nComputer-Aided\nEn